In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Data

In [2]:
df_train = pd.read_csv('data/train.csv')
df_test = pd.read_csv('data/test.csv')
df_original = pd.read_csv('data/Rainfall.csv')

### Train data

In [ ]:
df_train.head()

- we remove: 'id', 'day'
- 'rainfall' is our target

In [3]:
train = df_train.copy()
train.drop(['id', 'day'], axis=1, inplace=True)
train.head()

,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,windspeed,rainfall
0,1017.4,21.2,20.6,19.9,19.4,87.0,88.0,1.1,60.0,17.2,1
1,1019.5,16.2,16.9,15.8,15.4,95.0,91.0,0.0,50.0,21.9,1
2,1024.1,19.4,16.1,14.6,9.3,75.0,47.0,8.3,70.0,18.1,1
3,1013.4,18.1,17.8,16.9,16.8,95.0,95.0,0.0,60.0,35.6,1
4,1021.8,21.3,18.4,15.2,9.6,52.0,45.0,3.6,40.0,24.8,0


In [4]:
X_train = train.drop('rainfall', axis=1)
Y_train = train['rainfall']
target = 'rainfall'

### Test data

In [8]:
df_test.head()

,id,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,windspeed
0,2190,1,1019.5,17.5,15.8,12.7,14.9,96.0,99.0,0.0,50.0,24.3
1,2191,2,1016.5,17.5,16.5,15.8,15.1,97.0,99.0,0.0,50.0,35.3
2,2192,3,1023.9,11.2,10.4,9.4,8.9,86.0,96.0,0.0,40.0,16.9
3,2193,4,1022.9,20.6,17.3,15.2,9.5,75.0,45.0,7.1,20.0,50.6
4,2194,5,1022.2,16.1,13.8,6.4,4.3,68.0,49.0,9.2,20.0,19.4


In [9]:
test = df_test.copy()
test.drop(['id', 'day'], axis=1, inplace=True)
test.head()

,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,windspeed
0,1019.5,17.5,15.8,12.7,14.9,96.0,99.0,0.0,50.0,24.3
1,1016.5,17.5,16.5,15.8,15.1,97.0,99.0,0.0,50.0,35.3
2,1023.9,11.2,10.4,9.4,8.9,86.0,96.0,0.0,40.0,16.9
3,1022.9,20.6,17.3,15.2,9.5,75.0,45.0,7.1,20.0,50.6
4,1022.2,16.1,13.8,6.4,4.3,68.0,49.0,9.2,20.0,19.4


### original data

In [10]:
df_original.columns= df_original.columns.str.strip()
original = df_original.copy()
original.drop(['day'], axis=1, inplace=True)
original['rainfall'] = original['rainfall'].map({'no': 0, 'yes': 1})

In [11]:
original.head()

,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,rainfall,sunshine,winddirection,windspeed
0,1025.9,19.9,18.3,16.8,13.1,72,49,1,9.3,80.0,26.3
1,1022.0,21.7,18.9,17.2,15.6,81,83,1,0.6,50.0,15.3
2,1019.7,20.3,19.3,18.0,18.4,95,91,1,0.0,40.0,14.2
3,1018.9,22.3,20.6,19.1,18.8,90,88,1,1.0,50.0,16.9
4,1015.9,21.3,20.7,20.2,19.9,95,81,1,0.0,40.0,13.7


In [12]:
X_original = original.drop('rainfall', axis=1)
Y_original = original['rainfall']

# Combinations of columns

In [13]:
import itertools

In [14]:
# Zakładamy, że 'X_train' jest już załadowanym DataFrame
# Utworzymy słownik do przechowywania nowych kolumn
new_columns = {}

# Iterujemy przez wszystkie pary kolumn (bez powtórzeń)
for col1, col2 in itertools.combinations(X_train.columns, 2):
    # Dodawanie (kolejność nie ma znaczenia)
    new_columns[f'{col1}_plus_{col2}'] = X_train[col1] + X_train[col2]

    # Odejmowanie - wykonujemy w obu kierunkach
    new_columns[f'{col1}_minus_{col2}'] = X_train[col1] - X_train[col2]
    new_columns[f'{col2}_minus_{col1}'] = X_train[col2] - X_train[col1]

    # Mnożenie (kolejność nie ma znaczenia)
    new_columns[f'{col1}_times_{col2}'] = X_train[col1] * X_train[col2]

    # Dzielenie - wykonujemy w obu kierunkach
    new_columns[f'{col1}_div_{col2}'] = X_train[col1] / X_train[col2]
    new_columns[f'{col2}_div_{col1}'] = X_train[col2] / X_train[col1]

# Zamieniamy słownik na DataFrame
df_new = pd.DataFrame(new_columns, index=X_train.index)

# Opcjonalnie: łączymy oryginalny DataFrame z nowo utworzonymi kolumnami
df_comb = pd.concat([X_train, df_new], axis=1)

# Wyświetlamy wynik
df_comb.head()

,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,windspeed,...,windspeed_minus_sunshine,sunshine_times_windspeed,sunshine_div_windspeed,windspeed_div_sunshine,winddirection_plus_windspeed,winddirection_minus_windspeed,windspeed_minus_winddirection,winddirection_times_windspeed,winddirection_div_windspeed,windspeed_div_winddirection
0,1017.4,21.2,20.6,19.9,19.4,87.0,88.0,1.1,60.0,17.2,...,16.1,18.92,0.063953,15.636364,77.2,42.8,-42.8,1032.0,3.488372,0.286667
1,1019.5,16.2,16.9,15.8,15.4,95.0,91.0,0.0,50.0,21.9,...,21.9,0.00,0.000000,inf,71.9,28.1,-28.1,1095.0,2.283105,0.438000
2,1024.1,19.4,16.1,14.6,9.3,75.0,47.0,8.3,70.0,18.1,...,9.8,150.23,0.458564,2.180723,88.1,51.9,-51.9,1267.0,3.867403,0.258571
3,1013.4,18.1,17.8,16.9,16.8,95.0,95.0,0.0,60.0,35.6,...,35.6,0.00,0.000000,inf,95.6,24.4,-24.4,2136.0,1.685393,0.593333
4,1021.8,21.3,18.4,15.2,9.6,52.0,45.0,3.6,40.0,24.8,...,21.2,89.28,0.145161,6.888889,64.8,15.2,-15.2,992.0,1.612903,0.620000


### checking importance of new columns

we got some inf values, we need to replace them

In [16]:
# Zamiana wartości inf i -inf na NaN
df_comb.replace([np.inf, -np.inf], np.nan, inplace=True)

# Wypełnienie NaN zerami (lub możesz użyć dropna(), jeśli wolisz usunąć te wiersze)
df_comb.fillna(0, inplace=True)

In [17]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(random_state=42)
clf.fit(df_comb, Y_train)

# Pobieramy istotność cech
importances = clf.feature_importances_
features_importance = pd.Series(importances, index=df_comb.columns).sort_values(ascending=False)

print(features_importance)

dewpoint_plus_cloud                0.036983
cloud_minus_sunshine               0.033315
pressure_minus_cloud               0.031774
cloud_div_pressure                 0.029295
cloud_minus_pressure               0.027895
                                     ...   
windspeed_div_temparature          0.001065
maxtemp_minus_winddirection        0.001051
maxtemp_times_winddirection        0.001004
temparature_times_winddirection    0.001000
winddirection                      0.000569
Length: 280, dtype: float64


Let's check importance also for original data

In [18]:
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, Y_train)

# Pobieramy istotność cech
importances = clf.feature_importances_
features_importance = pd.Series(importances, index=X_train.columns).sort_values(ascending=False)

print(features_importance)

cloud            0.289859
sunshine         0.187422
humidity         0.098049
dewpoint         0.066691
windspeed        0.065287
pressure         0.065258
maxtemp          0.065230
mintemp          0.061798
temparature      0.057082
winddirection    0.043324
dtype: float64


Let's add 3 first new columns (dewpoint_plus_cloud, cloud_minus_sunshine, pressure minus cloud)  to original data

In [20]:
df_concat = pd.concat([X_original, df_new[['dewpoint_plus_cloud', 'cloud_minus_sunshine', 'pressure_minus_cloud']]], axis=1)
df_concat.head()

,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,windspeed,dewpoint_plus_cloud,cloud_minus_sunshine,pressure_minus_cloud
0,1025.9,19.9,18.3,16.8,13.1,72.0,49.0,9.3,80.0,26.3,107.4,86.9,929.4
1,1022.0,21.7,18.9,17.2,15.6,81.0,83.0,0.6,50.0,15.3,106.4,91.0,928.5
2,1019.7,20.3,19.3,18.0,18.4,95.0,91.0,0.0,40.0,14.2,56.3,38.7,977.1
3,1018.9,22.3,20.6,19.1,18.8,90.0,88.0,1.0,50.0,16.9,111.8,95.0,918.4
4,1015.9,21.3,20.7,20.2,19.9,95.0,81.0,0.0,40.0,13.7,54.6,41.4,976.8


In [23]:
clf = RandomForestClassifier(random_state=42)
clf.fit(df_concat, Y_train)

# Pobieramy istotność cech
importances = clf.feature_importances_
features_importance = pd.Series(importances, index=df_concat.columns).sort_values(ascending=False)

print(features_importance)

dewpoint_plus_cloud     0.349109
pressure_minus_cloud    0.288284
cloud_minus_sunshine    0.266701
mintemp                 0.011116
temparature             0.011063
pressure                0.011016
windspeed               0.010488
humidity                0.010089
maxtemp                 0.009854
cloud                   0.009500
dewpoint                0.008719
sunshine                0.007714
winddirection           0.006346
dtype: float64
